In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

In [2]:
@torch.no_grad()
def predict_tta(model: nn.Module, loader: DataLoader, device: torch.device, iterations: int = 2) -> torch.Tensor:
    # Переводим модель в режим инференса
    model.eval()
    model.to(device)
    
    # Список, где будем хранить итоговые тензоры логитов размером [N, C] для каждой итерации
    iteration_logits = []
    
    # 1. Запускаем цикл по количеству итераций
    for _ in range(iterations):
        single_prediction = []
        
        # 2. Внутри цикла проитерируемся по даталоадеру
        for x, _ in loader:
            x = x.to(device)
            
            # Получаем сырые выходы нейросети (логиты)
            logits = model(x)
            single_prediction.append(logits)
            
        # 3. Записываем ответы в один большой тензор размера [N, C] для текущей итерации
        # Используем конструкцию по подсказке ко второму скриншоту
        full_iteration_logits = torch.vstack(single_prediction)
        iteration_logits.append(full_iteration_logits)
        
    # 4. Собираем тензоры всех итераций в один огромный тензор
    # Сначала упаковываем в [iterations, N, C], затем меняем оси на [N, C, iterations]
    stacked_logits = torch.stack(iteration_logits, dim=0) # Размерность: [iterations, N, C]
    permuted_logits = stacked_logits.permute(1, 2, 0)     # Размерность: [N, C, iterations]
    
    # Усредняем его по оси итераций (dim=2), получая тензор размера [N, C]
    mean_logits = permuted_logits.mean(dim=2)
    
    # 5. Предсказываем классы как аргмакс по выходам вдоль оси классов (dim=1)
    final_classes = torch.argmax(mean_logits, dim=1)
    
    return final_classes

In [3]:
class AdvancedSkipConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # --- БЛОК 1: 32x32 -> 16x16 ---
        self.b1_conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.b1_bn1 = nn.BatchNorm2d(64)
        self.b1_relu1 = nn.ReLU()
        
        # Слой для skip-connection (вход 64 -> выход 64)
        self.b1_conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.b1_bn2 = nn.BatchNorm2d(64)
        self.b1_relu2 = nn.ReLU()
        
        self.b1_pool = nn.MaxPool2d(2)
        self.b1_drop = nn.Dropout(0.2)

        # --- БЛОК 2: 16x16 -> 8x8 ---
        self.b2_conv1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.b2_bn1 = nn.BatchNorm2d(128)
        self.b2_relu1 = nn.ReLU()
        
        # Слой для skip-connection (вход 128 -> выход 128)
        self.b2_conv2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.b2_bn2 = nn.BatchNorm2d(128)
        self.b2_relu2 = nn.ReLU()
        
        self.b2_pool = nn.MaxPool2d(2)
        self.b2_drop = nn.Dropout(0.3)

        # --- БЛОК 3: 8x8 -> 4x4 ---
        self.b3_conv1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.b3_bn1 = nn.BatchNorm2d(256)
        self.b3_relu1 = nn.ReLU()
        
        # Слой для skip-connection (вход 256 -> выход 256)
        self.b3_conv2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.b3_bn2 = nn.BatchNorm2d(256)
        self.b3_relu2 = nn.ReLU()
        
        self.b3_pool = nn.MaxPool2d(2)
        self.b3_drop = nn.Dropout(0.4)

        # --- Классификатор ---
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(4096, 512)
        self.fc_bn = nn.BatchNorm1d(512)
        self.fc_relu = nn.ReLU()
        self.fc_drop = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        # Блок 1
        x = self.b1_relu1(self.b1_bn1(self.b1_conv1(x)))
        # Добавляем skip-connection: x = block(x) + x
        x = self.b1_relu2(self.b1_bn2(self.b1_conv2(x))) + x
        x = self.b1_drop(self.b1_pool(x))

        # Блок 2
        x = self.b2_relu1(self.b2_bn1(self.b2_conv1(x)))
        # Добавляем skip-connection
        x = self.b2_relu2(self.b2_bn2(self.b2_conv2(x))) + x
        x = self.b2_drop(self.b2_pool(x))

        # Блок 3
        x = self.b3_relu1(self.b3_bn1(self.b3_conv1(x)))
        # Добавляем skip-connection
        x = self.b3_relu2(self.b3_bn2(self.b3_conv2(x))) + x
        x = self.b3_drop(self.b3_pool(x))

        # Выпрямление и классификация
        x = self.flatten(x)
        x = self.fc_drop(self.fc_relu(self.fc_bn(self.fc1(x))))
        x = self.fc2(x)
        return x

def create_advanced_skip_connection_conv_cifar() -> nn.Module:
    """Обязательная функция по условию задачи, возвращающая модель."""
    return AdvancedSkipConvNet()

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
# 1. Трансформации
means, stds = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(means, stds)
])

# Сильное условие задачи: создаем ТЕСТОВЫЕ аугментации для работы TTA
test_tta_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=2, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(means, stds)
])

# 2. Датасеты и Лоадеры
train_dataset = datasets.CIFAR10(root='/kaggle/input/notebooks/maximzatsepin/input/cifar10', train=True, download=True, transform=train_transform)
test_tta_dataset = datasets.CIFAR10(root='/kaggle/input/notebooks/maximzatsepin/input/cifar10', train=False, download=True, transform=test_tta_transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)

# Для TTA крайне важно оставить shuffle=False, чтобы порядок объектов во всех итерациях совпадал!
test_tta_loader = DataLoader(test_tta_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

In [5]:
# Этот лоадер идет в evaluate_epoch для честного мониторинга:
# ЧИСТЫЕ трансформации для валидации (БЕЗ случайных искажений)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(means, stds)
])
test_dataset = datasets.CIFAR10(root='/kaggle/input/notebooks/maximzatsepin/input/cifar10', train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

In [6]:
def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = criterion(output, y)
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    return total_loss / len(loader), correct / total

In [7]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    return total_loss / len(loader), correct / total

In [8]:
model = create_advanced_skip_connection_conv_cifar().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), weight_decay=1e-4)

epochs = 60
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=4e-3, steps_per_epoch=len(train_loader), epochs=epochs
)

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer,scheduler , criterion, device)
    te_loss, te_acc = evaluate_epoch(model, test_loader, criterion, device)
    print(f"Эпоха {epoch} | Train Acc: {tr_acc*100:.2f}% | Test Acc: {te_acc*100:.2f}%")

    # Если преодолели порог, делаем предсказания и сохраняем по правилам
    if te_acc >= 0.886:
        print("Целевая метрика в 88.5% достигнута!")
        
        # Получаем итоговый тензор предсказаний длины N (10000 для тест-сета CIFAR10)
        predictions = predict_tta(model, test_tta_loader, device, iterations=3)
        
        # Сохраняем строго через torch.save на диск в текущую папку
        # Переносим на CPU для гарантированной совместимости с чекером
        torch.save(predictions.cpu(), 'predictions.pt')
        print("Тензор предсказаний успешно записан в файл 'predictions.pt'")
        break

Эпоха 1 | Train Acc: 35.72% | Test Acc: 35.54%
Эпоха 2 | Train Acc: 50.79% | Test Acc: 54.07%
Эпоха 3 | Train Acc: 59.08% | Test Acc: 60.57%
Эпоха 4 | Train Acc: 64.11% | Test Acc: 67.17%
Эпоха 5 | Train Acc: 67.54% | Test Acc: 71.42%
Эпоха 6 | Train Acc: 69.75% | Test Acc: 74.43%
Эпоха 7 | Train Acc: 72.08% | Test Acc: 76.35%
Эпоха 8 | Train Acc: 73.76% | Test Acc: 75.70%
Эпоха 9 | Train Acc: 75.38% | Test Acc: 76.72%
Эпоха 10 | Train Acc: 77.11% | Test Acc: 78.46%
Эпоха 11 | Train Acc: 78.04% | Test Acc: 79.82%
Эпоха 12 | Train Acc: 78.99% | Test Acc: 81.27%
Эпоха 13 | Train Acc: 80.33% | Test Acc: 78.23%
Эпоха 14 | Train Acc: 81.19% | Test Acc: 82.15%
Эпоха 15 | Train Acc: 81.93% | Test Acc: 83.10%
Эпоха 16 | Train Acc: 82.80% | Test Acc: 84.49%
Эпоха 17 | Train Acc: 83.68% | Test Acc: 84.56%
Эпоха 18 | Train Acc: 84.22% | Test Acc: 84.46%
Эпоха 19 | Train Acc: 84.72% | Test Acc: 85.55%
Эпоха 20 | Train Acc: 85.19% | Test Acc: 85.90%
Эпоха 21 | Train Acc: 85.80% | Test Acc: 83.84%
Э